In [ ]:

import os, json, shutil, tempfile, subprocess, sys, time
from pathlib import Path
from multiprocessing import shared_memory
import torch
report = {'purpose': 'runtime_capability_probe_not_benchmark', 'checks': {},
          'cpu_affinity': sorted(os.sched_getaffinity(0)),
          'tmpdir': tempfile.gettempdir(),
          'limits': {'tmp_disk_gib': shutil.disk_usage('/tmp').total / 1024**3,
                     'shm_gib': shutil.disk_usage('/dev/shm').total / 1024**3,
                     'ram_gib': int(Path('/sys/fs/cgroup/memory.max').read_text()) / 1024**3},
          'root_readonly': bool(os.statvfs('/').f_flag & os.ST_RDONLY),
          'job_storage_free_gib': shutil.disk_usage('/output/tmp').free / 1024**3,
          'cache_dirs': {k: os.environ.get(k) for k in ('TRITON_CACHE_DIR','TORCHINDUCTOR_CACHE_DIR')}}
assert report['cpu_affinity'] == [0,1,2,3]
assert report['limits']['tmp_disk_gib'] > 24 and report['limits']['shm_gib'] == 30
assert os.stat('/tmp').st_ino == os.stat('/output/tmp').st_ino
assert os.stat('/tmp').st_dev == os.stat('/output/tmp').st_dev
assert os.statvfs('/tmp').f_flag & os.ST_NOSUID
assert os.statvfs('/tmp').f_flag & os.ST_NODEV
assert report['limits']['ram_gib'] == 30 and report['root_readonly']
assert Path(tempfile.gettempdir()).resolve() == Path('/output/tmp')
assert shutil.disk_usage('/output/tmp').free > 3*1024**3

with tempfile.TemporaryFile(dir='/tmp') as f:
    block = b'\x5a' * 1024**2
    for _ in range(2048):
        f.write(block)
    f.flush(); os.fsync(f.fileno())
    assert f.tell() == 2048*1024**2
    f.seek(-16,2); assert f.read(16) == b'\x5a'*16
report['checks']['2048_mib_tempfile_in_job_storage'] = 'passed'

segment = shared_memory.SharedMemory(create=True, size=256*1024**2)
try:
    segment.buf[:] = b'\0' * segment.size
    child = "from multiprocessing import shared_memory; import sys; s=shared_memory.SharedMemory(name=sys.argv[1],track=False); s.buf[0]=37; s.buf[-1]=73; s.close()"
    subprocess.run([sys.executable,'-c',child,segment.name],check=True,timeout=20)
    assert segment.buf[0] == 37 and segment.buf[-1] == 73
finally:
    segment.close(); segment.unlink()
report['checks']['256_mib_shared_memory_child'] = 'passed'

class CpuSavedSquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.device = x.device
        cpu = torch.empty_like(x,device='cpu',pin_memory=True)
        cpu.copy_(x)
        ctx.save_for_backward(cpu)
        return x.square()
    @staticmethod
    def backward(ctx, grad):
        (cpu,) = ctx.saved_tensors
        return 2*cpu.to(ctx.device)*grad
x=torch.linspace(-1,1,262144,device='cuda',requires_grad=True)
y=CpuSavedSquare.apply(x)
y.sum().backward()
torch.testing.assert_close(y,x.detach().square())
torch.testing.assert_close(x.grad,2*x.detach())
report['checks']['custom_autograd_cpu_saved_activation'] = 'passed'

started=time.perf_counter()
compiled=torch.compile(lambda a:a.sin()+a.square(),fullgraph=True)
actual=compiled(x.detach())
torch.cuda.synchronize()
torch.testing.assert_close(actual,x.detach().sin()+x.detach().square(),rtol=1e-5,atol=1e-6)
report['checks']['torch_compile_gpu_correctness'] = 'passed'
report['compile_first_call_seconds']=time.perf_counter()-started
report['cuda_device']=torch.cuda.get_device_name(0)
for name,path in report['cache_dirs'].items():
    assert path and Path(path).is_relative_to('/output/tmp')
    report[name+'_file_count']=sum(p.is_file() for p in Path(path).rglob('*'))
report['torch_peak_allocated_mib']=torch.cuda.max_memory_allocated()/1024**2
Path(os.environ['SOUP_RESULTS'],'capability-report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
